## Import and install dependencies

In [34]:
%pip install evaluate transformers torch accelerate -q

In [35]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)

cuda


In [36]:
BASE_MODEL_NAME = "prajjwal1/bert-tiny"

num_labels = 2
id2label = {0:'BR', 1:'PT'}
label2id = {'BR':0, 'PT':1}

model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect i

In [37]:
model.config

BertConfig {
  "add_cross_attention": false,
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 128,
  "id2label": {
    "0": "BR",
    "1": "PT"
  },
  "initializer_range": 0.02,
  "intermediate_size": 512,
  "is_decoder": false,
  "label2id": {
    "BR": 0,
    "PT": 1
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 2,
  "num_hidden_layers": 2,
  "pad_token_id": 0,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

## Loading Dataset

In [38]:
from datasets import Dataset, DatasetDict, load_dataset

SUBSET_SIZE = 1000

pt_stream = load_dataset('bastao/VeraCruz_PT-BR', "Portugal (PT)", split='train', streaming=True)
ptbr_stream = load_dataset('bastao/VeraCruz_PT-BR', "Brazil (BR)", split='train', streaming=True)
other_stream = load_dataset('bastao/VeraCruz_PT-BR', "Other", split='train', streaming=True)

pt_subset = pt_stream.take(SUBSET_SIZE)
ptbr_subset = ptbr_stream.take(SUBSET_SIZE)
other_subset = other_stream.take(SUBSET_SIZE)

def stream_to_dataset(stream):
    return Dataset.from_list(list(stream))

VERA_CRUZ_DATASETS = DatasetDict({
    "pt": stream_to_dataset(pt_subset),
    "ptbr": stream_to_dataset(ptbr_subset),
    "other": stream_to_dataset(other_subset),
})

VERA_CRUZ_DATASETS

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

DatasetDict({
    pt: Dataset({
        features: ['text', 'timestamp', 'url', 'source'],
        num_rows: 1000
    })
    ptbr: Dataset({
        features: ['text', 'timestamp', 'url', 'source'],
        num_rows: 1000
    })
    other: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'label', 'score'],
        num_rows: 1000
    })
})

## Tokenize

In [39]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

def preprocess(sample):
    return tokenizer(sample['text'], truncation=True)

# TOKENIZED_VERA_CRUZ_DATASETS = VERA_CRUZ_DATASETS.map(preprocess, batched=True)

# Trunacting max_length=512
TOKENIZED_VERA_CRUZ_DATASETS = VERA_CRUZ_DATASETS.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=512),
    batched=True
)

TOKENIZED_VERA_CRUZ_DATASETS

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    pt: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
    ptbr: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
    other: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'label', 'score', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
})

In [40]:
TOKENIZED_VERA_CRUZ_DATASETS["other"]

Dataset({
    features: ['text', 'timestamp', 'url', 'source', 'label', 'score', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1000
})

## Fine-tuning

In [41]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

label_map = {"BR": 0, "PT": 1} # Adjust based on your actual classes

def preprocess_labels(example):
    # Replace 'label' with the actual name of your label column
    example["label"] = label_map[example["label"]]
    return example

LABELED_DATASET = TOKENIZED_VERA_CRUZ_DATASETS['other'].map(preprocess_labels)

training_args = TrainingArguments(
    output_dir='teste',
    per_device_train_batch_size=64,
    logging_strategy='epoch',
    num_train_epochs=3,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=LABELED_DATASET,
    # eval_dataset=tokenized_imdb["test"],
    data_collator=data_collator,
    # compute_metrics=compute_metrics,
)


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [42]:
trainer.train()

Step,Training Loss
16,0.609940
32,0.564771
48,0.553370


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=48, training_loss=0.576026995976766, metrics={'train_runtime': 15.3744, 'train_samples_per_second': 195.129, 'train_steps_per_second': 3.122, 'total_flos': 3811461120000.0, 'train_loss': 0.576026995976766, 'epoch': 3.0})

## Evaluation

In [45]:
from sklearn.metrics import accuracy_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return accuracy_score(labels, preds)

preds = trainer.predict(LABELED_DATASET)
compute_metrics(preds)

0.758